# Merge monthly Chuuchuu summary stats

Vertically appends every `summary_stats/chuuchuu_summary_YYYY_MM.xlsx` into a single Excel.
Each sheet (`Summary`, `SNCF`, `DB`, ...) is merged across months separately, so the output keeps the same sheets and the same columns as the originals.

In [1]:
import re
from pathlib import Path

import pandas as pd

SUMMARY_DIR = Path("summary_stats")
OUTPUT_FILE = SUMMARY_DIR / "chuuchuu_summary_monthly_merged.xlsx"

# Only the monthly files (excludes *_full, *_french, etc.)
pattern = re.compile(r"^chuuchuu_summary_(\d{4})_(\d{2})\.xlsx$")
monthly_files = sorted(f for f in SUMMARY_DIR.iterdir() if pattern.match(f.name))
print(f"{len(monthly_files)} monthly files found:")
for f in monthly_files:
    print(" -", f.name)

12 monthly files found:
 - chuuchuu_summary_2025_01.xlsx
 - chuuchuu_summary_2025_02.xlsx
 - chuuchuu_summary_2025_03.xlsx
 - chuuchuu_summary_2025_04.xlsx
 - chuuchuu_summary_2025_05.xlsx
 - chuuchuu_summary_2025_06.xlsx
 - chuuchuu_summary_2025_07.xlsx
 - chuuchuu_summary_2025_08.xlsx
 - chuuchuu_summary_2025_09.xlsx
 - chuuchuu_summary_2025_10.xlsx
 - chuuchuu_summary_2025_11.xlsx
 - chuuchuu_summary_2025_12.xlsx


In [2]:
# Read every sheet of every file, keeping the sheet order and column order of the first file
merged = {}
reference_columns = {}

for f in monthly_files:
    sheets = pd.read_excel(f, sheet_name=None)
    for sheet_name, df in sheets.items():
        if sheet_name not in reference_columns:
            reference_columns[sheet_name] = list(df.columns)
        elif list(df.columns) != reference_columns[sheet_name]:
            print(f"WARNING: column mismatch in {f.name} / {sheet_name}")
        merged.setdefault(sheet_name, []).append(df)

merged = {
    sheet_name: pd.concat(dfs, ignore_index=True)[reference_columns[sheet_name]]
    for sheet_name, dfs in merged.items()
}

for sheet_name, df in merged.items():
    print(f"{sheet_name:<12} {df.shape}")

Summary      (2362, 21)
SNCF         (288, 22)
DB           (687, 22)
NS           (163, 22)
OEBB         (647, 22)
SBB          (401, 22)
SNCB         (194, 22)
Trenitalia   (394, 22)


In [3]:
# Sanity check: merged row count = sum of monthly row counts, all months present
for sheet_name, df in merged.items():
    expected = sum(len(pd.read_excel(f, sheet_name=sheet_name)) for f in monthly_files)
    assert len(df) == expected, f"{sheet_name}: {len(df)} rows vs {expected} expected"

merged["Summary"].groupby(["Year", "Month"]).size()

Year  Month
2025  1        220
      2        186
      3        177
      4        214
      5        188
      6        183
      7        178
      8        184
      9        185
      10       211
      11       209
      12       227
dtype: int64

In [4]:
with pd.ExcelWriter(OUTPUT_FILE) as writer:
    for sheet_name, df in merged.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved to {OUTPUT_FILE}")

Saved to summary_stats\chuuchuu_summary_monthly_merged.xlsx
